In [ ]:
"""
TRL GRPO Training for Sokoban — mirrors MLX GRPO notebook structure.
Model: LiquidAI/LFM2-1.2B-Tool

Structure follows the MLX notebook exactly:
  1. Config
  2. Load puzzles
  3. Build training dataset
  4. Build validation dataset
  5. Train
  6. Plot results
  7. Save final model
"""

import numpy as np
import matplotlib.pyplot as plt

from core.microban import load_microban
from rl import build_dataset
from rl.grpo import GRPOConfig, train_grpo, debug_generation, save_merged_model

print("=" * 60)
print("TRL GRPO for Sokoban  —  LFM2-1.2B-Tool")
print("=" * 60)


# ============================================================================
# 1. Configuration
# ============================================================================

config = GRPOConfig(
    model_id="LiquidAI/LFM2-1.2B-Tool",

    # Data
    repr_key="13_ACTION_CENTRIC",
    num_train_puzzles=10,
    max_steps_per_puzzle=10,

    # GRPO (matching MLX defaults)
    num_generations=4,          # MLX group_size=4
    learning_rate=1e-5,         # MLX learning_rate=1e-5
    beta=0.02,                  # MLX beta=0.02
    epsilon=0.2,                # MLX epsilon=0.2
    max_completion_length=265,  # MLX max_response_len=265
    num_epochs=1.0,
    per_device_batch_size=2,    # MLX batch_size=2
    gradient_accumulation_steps=4,

    # LoRA (matching MLX)
    lora_r=8,                   # MLX lora_rank=8
    lora_alpha=10,              # MLX lora_scale=10
    lora_dropout=0.0,           # MLX lora_dropout=0.0

    # LFM2 is small — no need for 4-bit on most hardware; flip to True if OOM
    load_in_4bit=False,

    output_dir="./rl-output",
    logging_steps=10,
    save_steps=100,
    seed=42,
)


# ============================================================================
# 2. Load puzzles
# ============================================================================

puzzles = load_microban("Microban.txt")
train_puzzles = [p for p in puzzles if p.num_boxes <= 3][: config.num_train_puzzles]
val_puzzles = [p for p in puzzles if p.num_boxes <= 3][
    config.num_train_puzzles : config.num_train_puzzles + 10
]

print(f"\nTraining puzzles:   {len(train_puzzles)}")
print(f"Validation puzzles: {len(val_puzzles)}")


# ============================================================================
# 3. Build training dataset
# ============================================================================

print("\nBuilding training dataset...")
hf_train_dataset = build_dataset(
    puzzles=train_puzzles,
    repr_key=config.repr_key,           # from GRPOConfig — mirrors MLX config.repr_key
    max_steps_per_puzzle=config.max_steps_per_puzzle,
    verbose=True,
)

print(f"  Total training examples: {len(hf_train_dataset)}")

# Sanity-check the first example (mirrors MLX notebook check)
sample = hf_train_dataset[0]
print("=" * 60)
print("SAMPLE PROMPT (first 500 chars):")
print("=" * 60)
print(sample["prompt"][:500])
print("...")
print("\nDoes it contain the board?")
print("  '####' in prompt:", "####" in sample["prompt"])
print("  '@'    in prompt:", "@" in sample["prompt"])
print("  '$'    in prompt:", "$" in sample["prompt"])
print(f"\n  optimal_move:  {sample.get('optimal_move')}")
print(f"  optimal_cost:  {sample.get('optimal_cost')}")


# ============================================================================
# 4. Build validation dataset
# ============================================================================

print("\nBuilding validation dataset...")
hf_val_dataset = build_dataset(
    puzzles=val_puzzles,
    repr_key=config.repr_key,
    max_steps_per_puzzle=config.max_steps_per_puzzle,
    verbose=True,
)

print(f"  Total validation examples: {len(hf_val_dataset)}")


# ============================================================================
# 5. Debug generation before training (mirrors MLX notebook)
# ============================================================================

print("\n" + "=" * 60)
print("Starting GRPO Training")
print(f"  Model:        {config.model_id}")
print(f"  Iterations:   {config.num_epochs} epochs")
print(f"  Group size:   {config.num_generations}")
print(f"  Batch size:   {config.per_device_batch_size}")
print(f"  Learning rate:{config.learning_rate}")
print(f"  beta (KL):    {config.beta}")
print(f"  epsilon:      {config.epsilon}")
print("=" * 60 + "\n")

# Load model temporarily for pre-training debug generation.
# train_grpo will reload it internally — this just mirrors what the
# MLX notebook does before calling grpo_train_loop.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

_tokenizer = AutoTokenizer.from_pretrained(config.model_id)
if _tokenizer.pad_token_id is None:
    _tokenizer.pad_token_id = _tokenizer.eos_token_id

_model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
_model.eval()

debug_generation(_model, _tokenizer, sample["prompt"])

# Free memory before training loads its own copy
del _model
torch.cuda.empty_cache()


# ============================================================================
# 6. Train
# ============================================================================

trainer = train_grpo(
    dataset=hf_train_dataset,
    config=config,
    eval_dataset=hf_val_dataset,
)


# ============================================================================
# 7. Plot results  (mirrors MLX notebook plot)
# ============================================================================

log_history = trainer.state.log_history

# Extract loss and reward from TRL log history
losses  = [e["loss"]          for e in log_history if "loss"   in e]
rewards = [e["train/reward"]  for e in log_history if "train/reward" in e]

if losses and rewards:
    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss", color="tab:red")
    ax1.plot(losses, color="tab:red", alpha=0.7)
    ax1.tick_params(axis="y", labelcolor="tab:red")

    window = min(20, len(losses))
    smoothed_loss = np.convolve(losses, np.ones(window) / window, mode="valid")
    ax1.plot(
        range(window - 1, len(losses)),
        smoothed_loss,
        color="darkred",
        linewidth=2,
        label="Smoothed loss",
    )

    ax2 = ax1.twinx()
    ax2.set_ylabel("Reward", color="tab:blue")
    n = min(20, len(rewards))
    moving_avg = np.convolve(rewards, np.ones(n) / n, mode="valid")
    ax2.plot(
        range(n - 1, len(rewards)),
        moving_avg,
        color="tab:blue",
        linewidth=2,
        label="Reward (MA)",
    )
    ax2.tick_params(axis="y", labelcolor="tab:blue")

    plt.title("GRPO Training: Loss vs Reward")
    fig.tight_layout()

    import os
    os.makedirs(config.output_dir, exist_ok=True)
    curve_path = os.path.join(config.output_dir, "training_curves.png")
    plt.savefig(curve_path, dpi=150)
    plt.show()
    print(f"\n✓ Training curves saved to {curve_path}")
else:
    print("\n⚠ No loss/reward entries found in log history — skipping plot.")


# ============================================================================
# 8. Save final model
# ============================================================================

# adapter_path = os.path.join(config.output_dir, "adapters")
# os.makedirs(adapter_path, exist_ok=True)

# # Save LoRA adapters (light, shareable)
# trainer.model.save_pretrained(adapter_path)
# trainer.processing_class.save_pretrained(adapter_path)
# print(f"✓ LoRA adapters saved to {adapter_path}")

# # Optionally merge weights into a standalone model
# merged_path = os.path.join(config.output_dir, "merged")
# save_merged_model(trainer, merged_path)

TRL GRPO for Sokoban  —  LFM2-1.2B-Tool
  [microban] Skipping puzzle "155 'The Dungeon'": (8, 8)

Training puzzles:   10
Validation puzzles: 10

Building training dataset...
  Puzzle 1: 33 steps (8 pushes)
  Puzzle 2: 16 steps (3 pushes)
  Puzzle 3: 41 steps (13 pushes)
  Puzzle 4: 31 steps (7 pushes)
  Puzzle 6: 147 steps (29 pushes)
  Puzzle 8: 101 steps (32 pushes)
  Puzzle 9: 30 steps (10 pushes)
  Puzzle 10: 129 steps (21 pushes)
  Puzzle 11: 78 steps (16 pushes)
  Puzzle 12: 49 steps (11 pushes)

  Total training examples: 100
  Total training examples: 100
SAMPLE PROMPT (first 500 chars):
You are an expert Sokoban solver.
Legend: # = wall | @ = player | $ = box | . = goal | * = box-on-goal | + = player-on-goal | (space) = floor
Rules:
- Push boxes onto ALL goal squares to win.
- You PUSH boxes by walking into them — you cannot pull.
- A box cannot be pushed into a wall or another box.
- Move encoding (LURD):
    Uppercase = push a box:  U=push-up  D=push-down  L=push-left  R=pus

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cpu).
W0426 18:03:33.096000 24868 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


DEBUG: Model Generation

Raw response:
<NextMove>U</NextMove>
<Confidence>0.95</Confidence>
<Rationale>Push the unoccupied box towards tile 4 to block the goal, as this is necessary to clear the way for a potential win. It's the most direct way to progress towards the goal while adhering to the rules of Sokoban.</Rationale>

Has  thinking:    False
Has <NextMove>:   True
Has <Confidence>: True
Has <Rationale>:  True
Extracted move: U


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


  Starting GRPO training (MLX-aligned logic)
  Model:          LiquidAI/LFM2-1.2B-Tool
  Dataset:        100 examples
  Generations:    4 per prompt
  Epochs:         1.0
  Reward weights: format=0.3, progress=0.4, optimal=0.2, move_quality=0.1
  Clamp range:    [-1.0, 2.0]
  beta (KL):      0.02
  epsilon (clip): 0.2
  Output:         ./rl-output



c:\anaconda3\envs\fine-tune\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.006950
20,-0.124690
30,-0.056582
40,-0.063257
50,-0.153343



⚠ No loss/reward entries found in log history — skipping plot.


NameError: name 'os' is not defined

In [ ]:
from llm.predictor import LLMPredictor, TransformersBackend

base_backend = TransformersBackend(
    model_id="google/gemma-3-4b-it",
    torch_dtype="float16",
)
base_predictor = LLMPredictor(base_backend)



# Evaluation

In [1]:
from llm.predictor import LLMPredictor, TransformersBackend

# Base model
base_backend = TransformersBackend(
    model_id="LiquidAI/LFM2-1.2B-Tool",
    torch_dtype="float16",
)
base_predictor = LLMPredictor(base_backend)

# GRPO model (your trained one)
grpo_backend = TransformersBackend(
    model_id="./rl-output/merged",
    torch_dtype="float16",
)
grpo_predictor = LLMPredictor(grpo_backend)

In [2]:
from rl import quick_evaluation

results = quick_evaluation(
    base_predictor=base_predictor,
    grpo_predictor=grpo_predictor,
    num_test_puzzles=5,
)

  [microban] Skipping puzzle "155 'The Dungeon'": (8, 8)

📚 Test set: 5 puzzles
   Easy: 1
   Medium: 4
   Hard: 0

EVALUATING BASE MODEL
  [1/5] Evaluating 44 'Duh!'... 

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cpu).
W0426 22:07:32.768000 3264 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ (solved=True, pushes=1/1)
  [2/5] Evaluating 51... 

...

✗ (solved=False, pushes=0/10)
  [4/5] Evaluating 79... 

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

✗ (solved=False, pushes=0/18)
  [5/5] Evaluating 14... 

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

✓ (solved=True, pushes=10/10)

EVALUATION SUMMARY

📊 Overall Performance:
            Solve Rate  Solved  Reward  Move Quality  Format  LLM Calls  Time (s)  Avg Pushes  Optimal Pushes
model_type                                                                                                   
base              60.0       3   0.187         0.115    0.24       29.8  1061.547         3.8             9.4
grpo              60.0       3   0.187         0.115    0.24       29.8  1060.050         3.8             9.4

📊 Performance by Difficulty:
                       solved  reward_score
model_type difficulty                      
base       easy         100.0         2.120
           medium        50.0        -0.296
grpo       easy         100.0         2.120
           medium        50.0        -0.296

📈 Improvement:
  Base solve rate: 60.0%
  GRPO solve rate: 60.0%
  Improvement: +0.0%

📊 Per-Puzzle Comparison (Base → GRPO):
  Puzzles where GRPO improved: 0
  Puzzles where GRPO regressed: 

In [2]:
%pip install -U transformers

  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 699.0 kB/s eta 0:00:15
   -- ------------------------------------- 0.5/10.4 MB 699.0 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/10.4 MB 699.0 kB/s eta 0:00:14
   --- ------------------------------------ 0.8/10.4 MB 699.0 kB/s eta 0:00:14
   ---- ----------------------------------- 1.0/10.4 MB 699.0 kB/s eta 0:00:14
   ----- ---------------------------------- 1.3/10.4 MB 706.6 kB/s eta 0:00:13
   ----- ---------------------------------- 1.3/10.4 MB 706.6 kB/s eta 0:00:13
   ------ ---------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.10.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,<=4.56.2,>=4.51.3, but you have transformers 5.6.2 which is incompatible.


In [2]:
import os

adapter_path = os.path.join(config.output_dir, "adapters")
os.makedirs(adapter_path, exist_ok=True)

# Save LoRA adapters (light, shareable)
trainer.model.save_pretrained(adapter_path)
trainer.processing_class.save_pretrained(adapter_path)
print(f"✓ LoRA adapters saved to {adapter_path}")

# Optionally merge weights into a standalone model
merged_path = os.path.join(config.output_dir, "merged")
save_merged_model(trainer, merged_path)


✓ LoRA adapters saved to ./rl-output\adapters
  Merging LoRA weights and saving to ./rl-output\merged ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Saved to ./rl-output\merged


In [1]:
%pip install triton accelerate

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement triton (from versions: none)
ERROR: No matching distribution found for triton


In [1]:
from transformers.models.lfm2.modeling_lfm2 import Lfm2ForCausalLM

In [2]:
%pip install --upgrade torch torchvision --extra-index-url https://pytorch.org

^C
Note: you may need to restart the kernel to use updated packages.
